# Dataset inspection

Displays machine-readable M2 inspection artifacts without decoding the full video corpus.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

roots = [Path("artifacts/validation/M2"), Path("artifacts/dataset_inspection")]
candidates = []
for root in roots:
    if root.exists():
        candidates.extend(root.rglob("inspection.json"))
artifact = max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None
if artifact is None:
    display(Markdown("M2 inspection artifact is not available yet. Run `scripts/inspect_dataset.py`."))
else:
    report = json.loads(artifact.read_text())
    md = artifact.with_name("inspection.md")
    display(Markdown(f"Loaded `{artifact}`"))
    display(Markdown(md.read_text() if md.exists() else f"acceptance_complete={report['acceptance_complete']}"))

In [ ]:
if artifact is not None:
    seen = report["suites"]["libero_90"]
    goal = report["suites"]["libero_goal"]
    display(Markdown(
        f"| suite | episodes | frames | tasks | fps |\n"
        f"|---|---:|---:|---:|---:|\n"
        f"| libero_90 | {seen['episodes']} | {seen['frames']} | {seen['unique_task_text_count']} | {seen['fps']} |\n"
        f"| libero_goal | {goal['episodes']} | {goal['frames']} | {goal['unique_task_text_count']} | {goal['fps']} |"
    ))
    display(Markdown("Target first-25 episode IDs:"))
    display(report["target_episode_ids"])
    failed = [check for check in report["checks"] if check["status"] != "pass"]
    display(Markdown("All checks passed." if not failed else f"Failed checks: {failed}"))

In [ ]:
# Optional single-frame preview. M2 does not decode the video corpus.
frames = []
if Path("artifacts").exists():
    frames = [
        path
        for path in Path("artifacts").rglob("*")
        if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ]
if frames:
    display(Image(filename=str(frames[0])))
else:
    display(Markdown(
        "No sample frame is stored yet. Videos are downloaded only with "
        "`--include-videos` and decoded later for expert replay (M3), not during inspection."
    ))